# 1. Imports

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaConfig, RobertaForSequenceClassification, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score, roc_auc_score
import matplotlib.pyplot as plt
from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
import random
import os
import warnings
from tqdm import tqdm
warnings.filterwarnings("ignore")


# 2. Configuration

In [ ]:

SEED = 42
MAX_LEN = 128
BATCH_SIZE = 16
LR = 3e-5
EPOCHS = 3
WARMUP_PROPORTION = 0.1
MIN_LR = 1e-6
WEIGHT_DECAY = 0.01
MODEL_NAME = 'roberta-base'
LOSS_FUNCTION = 'bce'
FOCAL_ALPHA = 1
FOCAL_GAMMA = 2
DATA_VERSION = 'balanced'
LABELS = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']


# 3. Utilities

In [ ]:

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()

def threshold_search(y_true, y_pred_probs):
    best_thresholds = []
    for i in range(y_true.shape[1]):
        best_thr, best_f1 = 0.5, 0
        for thr in np.linspace(0.05, 0.95, 19):
            preds = (y_pred_probs[:, i] > thr).astype(int)
            score = f1_score(y_true[:, i], preds)
            if score > best_f1:
                best_f1 = score
                best_thr = thr
        best_thresholds.append(best_thr)
    return np.array(best_thresholds)

class FocalLoss(nn.Module):
    def __init__(self, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        bce_loss = nn.BCEWithLogitsLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        else:
            return focal_loss.sum()


# 4. Dataset and Dataloader

In [ ]:

class ToxicDataset(Dataset):
    def __init__(self, comments, labels, tokenizer, max_len):
        self.comments = comments
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.comments)

    def __getitem__(self, idx):
        comment = str(self.comments[idx])
        encoding = self.tokenizer(comment, truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.FloatTensor(self.labels[idx])
        }


# 5. Model

In [ ]:

class ToxicClassifier(nn.Module):
    def __init__(self):
        super(ToxicClassifier, self).__init__()
        config = RobertaConfig.from_pretrained(
            MODEL_NAME,
            num_labels=len(LABELS),
            problem_type='multi_label_classification',
            hidden_dropout_prob=0.1,
            attention_probs_dropout_prob=0.1
        )
        self.model = RobertaForSequenceClassification.from_pretrained(MODEL_NAME, config=config)

    def forward(self, input_ids, attention_mask):
        return self.model(input_ids=input_ids, attention_mask=attention_mask).logits


# 6. Training and Evaluation Functions

In [ ]:

def train_fn(model, data_loader, optimizer, scheduler, loss_fn, device):
    model.train()
    total_loss = 0
    all_labels, all_preds = [], []
    loop = tqdm(data_loader, desc='Training')
    for batch in loop:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.sigmoid(outputs).detach().cpu().numpy()
        all_preds.append(preds)
        all_labels.append(labels.cpu().numpy())

        loop.set_postfix(loss=loss.item())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)
    thresholds = threshold_search(all_labels, all_preds)
    preds_binary = (all_preds > thresholds).astype(int)

    return total_loss / len(data_loader), all_labels, all_preds, preds_binary

def eval_fn(model, data_loader, loss_fn, device, thresholds=None):
    model.eval()
    total_loss = 0
    all_labels, all_preds = [], []
    loop = tqdm(data_loader, desc='Evaluating')
    with torch.no_grad():
        for batch in loop:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs, labels)
            total_loss += loss.item()

            preds = torch.sigmoid(outputs).detach().cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    all_preds = np.vstack(all_preds)
    all_labels = np.vstack(all_labels)

    if thresholds is None:
        thresholds = threshold_search(all_labels, all_preds)

    preds_binary = (all_preds > thresholds).astype(int)

    return total_loss / len(data_loader), all_labels, all_preds, preds_binary

def compute_metrics(y_true, y_probs, y_pred_binary):
    metrics = {}
    metrics['macro_f1'] = f1_score(y_true, y_pred_binary, average='macro')
    metrics['macro_precision'] = precision_score(y_true, y_pred_binary, average='macro')
    metrics['macro_recall'] = recall_score(y_true, y_pred_binary, average='macro')
    metrics['macro_auc'] = roc_auc_score(y_true, y_probs, average='macro')

    per_class_metrics = {}
    for idx, label in enumerate(LABELS):
        per_class_metrics[label] = {
            'f1': f1_score(y_true[:, idx], y_pred_binary[:, idx]),
            'precision': precision_score(y_true[:, idx], y_pred_binary[:, idx]),
            'recall': recall_score(y_true[:, idx], y_pred_binary[:, idx]),
            'auc': roc_auc_score(y_true[:, idx], y_probs[:, idx])
        }
    return metrics, per_class_metrics


# 7. Main Execution

In [ ]:

def main():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    df = pd.read_csv("train.csv")

    if DATA_VERSION == 'balanced':
        toxic = df[df[LABELS].sum(axis=1) > 0]
        clean = df[df[LABELS].sum(axis=1) == 0].sample(n=len(toxic), random_state=SEED)
        df = pd.concat([toxic, clean]).sample(frac=1, random_state=SEED).reset_index(drop=True)

    y = df[LABELS].values
    mskf = MultilabelStratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    train_idx, val_idx = next(mskf.split(df['comment_text'], y))
    train_df = df.iloc[train_idx]
    val_df = df.iloc[val_idx]

    tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
    train_dataset = ToxicDataset(train_df['comment_text'].values, train_df[LABELS].values, tokenizer, MAX_LEN)
    val_dataset = ToxicDataset(val_df['comment_text'].values, val_df[LABELS].values, tokenizer, MAX_LEN)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = ToxicClassifier()
    model.to(device)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=int(WARMUP_PROPORTION * total_steps), num_training_steps=total_steps)

    loss_fn = FocalLoss() if LOSS_FUNCTION == 'focal' else nn.BCEWithLogitsLoss()

    train_losses, val_losses, val_f1s = [], [], []

    for epoch in range(EPOCHS):
        train_loss, y_train_true, y_train_probs, y_train_preds = train_fn(model, train_loader, optimizer, scheduler, loss_fn, device)
        val_loss, y_val_true, y_val_probs, y_val_preds = eval_fn(model, val_loader, loss_fn, device)

        train_metrics, train_per_class = compute_metrics(y_train_true, y_train_probs, y_train_preds)
        val_metrics, val_per_class = compute_metrics(y_val_true, y_val_probs, y_val_preds)

        print(f"Epoch {epoch+1}/{EPOCHS}")
        print(f"Train Loss: {train_loss:.4f} | Validation Loss: {val_loss:.4f}")
        print(f"Train Macro F1: {train_metrics['macro_f1']:.4f} | Validation Macro F1: {val_metrics['macro_f1']:.4f}")

    plt.figure(figsize=(10,6))
    plt.plot(range(1, EPOCHS+1), train_losses, label='Train Loss')
    plt.plot(range(1, EPOCHS+1), val_losses, label='Validation Loss')
    plt.xticks(range(1, EPOCHS+1))
    plt.legend()
    plt.title(f"Training and Validation Loss - {MODEL_NAME}_{LOSS_FUNCTION}_{DATA_VERSION}")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.grid()
    plt.show()

    plt.figure(figsize=(10,6))
    plt.plot(range(1, EPOCHS+1), val_f1s, label='Validation Macro F1', marker='o')
    plt.xticks(range(1, EPOCHS+1))
    plt.legend()
    plt.title(f"Validation Macro F1 per Epoch - {MODEL_NAME}_{LOSS_FUNCTION}_{DATA_VERSION}")
    plt.xlabel("Epoch")
    plt.ylabel("Macro F1")
    plt.grid()
    plt.show()

    torch.save(model.state_dict(), f"best_model_{MODEL_NAME}_{LOSS_FUNCTION}_{DATA_VERSION}.bin")
    final_thresholds = threshold_search(y_val_true, y_val_probs)
    np.save(f"best_thresholds_{MODEL_NAME}_{LOSS_FUNCTION}_{DATA_VERSION}.npy", final_thresholds)




In [ ]:
if __name__ == "__main__":
    main()